# PlantCLEF2026 Baseline with MLflow
This notebook reproduces the tiling inference baseline and tracks the experiment using the local MLflow and MinIO setup.

In [ ]:
import sys
import os

# Add the src folder to the Python path
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import timm
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import time
import torchvision.transforms as T
from torch.amp import autocast
from matplotlib import pyplot as plt
from kornia import tensor_to_image
from kornia.contrib import extract_tensor_patches, compute_padding
import csv
import mlflow

from src.config.mlflow_init import init_mlflow

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")

# Initialize MLflow experiment using the loaded config
init_mlflow()

In [ ]:
from src.utils.metrics import AverageMeter
from src.data.datasets import PatchDataset, TestDataset

In [ ]:
from src.data.metadata import load_metadata

data_dir = os.environ.get("DATA_DIR", "../data")
df_species_ids, df_metadata, class_map, reverse_class_map = load_metadata(data_dir)

df_metadata.head()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "vit_base_patch14_reg4_dinov2.lvd142m"
model_dir = os.environ.get("MODEL_DIR", "../model")

# Initialize model
model = timm.create_model(
    model_name,
    pretrained=False,
    num_classes=len(df_species_ids),
    checkpoint_path=f"{model_dir}/model_best.pth.tar",
)  # Directed to the extracted model folder
model = model.to(device)
model = model.eval()

data_config = timm.data.resolve_model_data_config(model)
model_input_size, model_mean, model_std = (
    data_config["input_size"][1],
    data_config["mean"],
    data_config["std"],
)

In [ ]:
# Hyperparameters
batch_size = 64
min_score = 0.1
top_k_tile = 2
patch_size = model_input_size
stride = int(model_input_size / 2)
use_pad = True

Validation Function

In [ ]:
def compute_val_metrics(model, composition_path, ground_truth_path, data_dir):
    """
    Valida usando quadrats sintéticos — sem precisar de imagens de quadrat reais.

    Para cada quadrat sintético:
      - pega as imagens individuais que o compõem
      - roda o classificador em cada imagem
      - agrega as espécies preditas (união com threshold)
      - compara com o ground truth
    """
    import ast
    from sklearn.metrics import f1_score, precision_score, recall_score
    from sklearn.preprocessing import MultiLabelBinarizer

    if not os.path.exists(composition_path) or not os.path.exists(ground_truth_path):
        print("Arquivos de validação não encontrados. Pulando métricas.")
        return None

    df_comp = pd.read_csv(composition_path)
    df_gt = pd.read_csv(ground_truth_path)

    def _parse(x):
        if isinstance(x, str):
            x = ast.literal_eval(x)
        return x if isinstance(x, list) else [x]

    df_gt["species_ids"] = df_gt["species_ids"].apply(_parse)

    transform_single = T.Compose(
        [
            T.Resize((model_input_size, model_input_size)),
            T.ToTensor(),
            T.Normalize(mean=model_mean, std=model_std),
        ]
    )

    predictions = {}

    quadrat_ids = df_comp["quadrat_id"].unique()
    print(f"Avaliando {len(quadrat_ids)} quadrats sintéticos...")

    for quadrat_id in quadrat_ids:
        group = df_comp[df_comp["quadrat_id"] == quadrat_id]
        quadrat_species = set()

        for _, row in group.iterrows():
            # monta caminho completo da imagem
            img_path = (
                os.path.join(data_dir, row["image_path"])
                if not os.path.isabs(row["image_path"])
                else row["image_path"]
            )

            if not os.path.exists(img_path):
                continue

            img = Image.open(img_path).convert("RGB")
            tensor = transform_single(img).unsqueeze(0).to(device)

            with torch.no_grad(), autocast(device.type):
                output = model(tensor)
                probs = torch.softmax(output, dim=1)
                top_probs, top_indices = torch.topk(probs, top_k_tile)

            for idx, prob in zip(top_indices[0], top_probs[0]):
                if prob.item() > min_score:
                    quadrat_species.add(class_map[idx.item()])

        predictions[quadrat_id] = list(quadrat_species)

    # calcula métricas
    df_pred = pd.DataFrame(
        [
            {"quadrat_id": qid, "predicted_species_ids": preds}
            for qid, preds in predictions.items()
        ]
    )

    df_eval = pd.merge(df_gt, df_pred, on="quadrat_id", how="inner")
    print(f"GT: {len(df_gt)} | Pred: {len(df_pred)} | Match: {len(df_eval)}")

    if df_eval.empty:
        print("Nenhum quadrat_id em comum. Verifique os caminhos das imagens.")
        return None

    mlb = MultiLabelBinarizer()
    mlb.fit(pd.concat([df_eval["species_ids"], df_eval["predicted_species_ids"]]))
    y_true = mlb.transform(df_eval["species_ids"])
    y_pred = mlb.transform(df_eval["predicted_species_ids"])

    return {
        "val_f1_samples": f1_score(y_true, y_pred, average="samples", zero_division=0),
        "val_precision_samples": precision_score(
            y_true, y_pred, average="samples", zero_division=0
        ),
        "val_recall_samples": recall_score(
            y_true, y_pred, average="samples", zero_division=0
        ),
    }

Submit prediction inside an MLflow run to log metrics and artifacts

In [ ]:
# Setup PyTorch Dataset and DataLoader
image_to_tensor = T.ToTensor()
data_dir = os.environ.get("DATA_DIR", "../data")
dataset = TestDataset(
    image_folder=f"{data_dir}/PlantCLEF2025_test_images/PlantCLEF2025_test_images/",
    patch_size=patch_size,
    stride=stride,
    use_pad=True,
    transform=image_to_tensor,
)
dataloader = DataLoader(dataset, batch_size=1, num_workers=4, pin_memory=True)

image_predictions = {}
image_probabilities = {}
batch_time = AverageMeter()

mlflow.pytorch.autolog()

with mlflow.start_run(run_name="tiling-inference-baseline") as run:

    mlflow.log_params(
        {
            "model_name": model_name,
            "batch_size": batch_size,
            "min_score": min_score,
            "top_k_tile": top_k_tile,
            "patch_size": patch_size,
            "stride": stride,
            "use_pad": use_pad,
            "device": device.type,
            "dataset_size": len(dataset),
        }
    )

    mlflow.pytorch.log_model(model, "vit_dino_model")

    # ==========================================
    # INFERÊNCIA NO CONJUNTO DE TESTE
    # ==========================================
    print("=== Inferência no conjunto de TESTE ===")
    end = time.time()

    with torch.no_grad():
        for batch_idx, (patches, image_path) in enumerate(dataloader):
            image_results = {}
            quadrat_id = os.path.splitext(os.path.basename(image_path[0]))[0]
            transform_patch = T.Normalize(mean=model_mean, std=model_std)
            patch_dataset = PatchDataset(patches[0], transform=transform_patch)
            patch_loader = DataLoader(
                patch_dataset, batch_size=batch_size, shuffle=False
            )

            for batch_patches in patch_loader:
                batch_patches = batch_patches.to(device)
                with autocast(device.type):
                    outputs = model(batch_patches)
                    probabilities = torch.nn.functional.softmax(outputs, dim=1)
                    top_probs, top_indices = torch.topk(probabilities, top_k_tile)
                    top_probs = top_probs.cpu().numpy()
                    top_indices = top_indices.cpu().numpy()

                    for top_tile_indices, top_tile_probs in zip(top_indices, top_probs):
                        for top_idx, top_prob in zip(top_tile_indices, top_tile_probs):
                            species_id = class_map[top_idx]
                            if top_prob > min_score:
                                if (
                                    species_id not in image_results
                                    or image_results[species_id] < top_prob
                                ):
                                    image_results[species_id] = top_prob

            predictions_list = list(image_results.keys())
            image_predictions[quadrat_id] = predictions_list
            image_probabilities[quadrat_id] = image_results

            num_predictions = len(predictions_list)
            max_prob = max(image_results.values()) if image_results else 0.0

            batch_time.update(time.time() - end)
            end = time.time()

            mlflow.log_metric("step_batch_time", batch_time.val, step=batch_idx)
            mlflow.log_metric(
                "num_predictions_per_image", num_predictions, step=batch_idx
            )
            mlflow.log_metric("max_prob_per_image", max_prob, step=batch_idx)

            if batch_idx % 10 == 0:
                print(
                    f"Predict: [{batch_idx}/{len(dataloader)}] Time {batch_time.val:.3f} ({batch_time.avg:.3f})"
                )

    mlflow.log_metric("avg_batch_time_seconds", batch_time.avg)
    mlflow.log_metric("total_images_processed", len(dataloader))

    # salva submission.csv
    submission_path = "submission.csv"
    df_submission = pd.DataFrame(
        list(image_predictions.items()), columns=["quadrat_id", "species_ids"]
    )
    df_submission["species_ids"] = df_submission["species_ids"].apply(str)
    df_submission.to_csv(submission_path, sep=",", index=False, quoting=csv.QUOTE_ALL)
    mlflow.log_artifact(submission_path)
    print(f"submission.csv salvo com {len(df_submission)} predições.")

    # ==========================================
    # VALIDAÇÃO COM QUADRATS SINTÉTICOS
    # ==========================================
    print("\n=== Validação com quadrats sintéticos ===")
    composition_path = f"{data_dir}/val_quadrat_composition.csv"
    ground_truth_path = f"{data_dir}/validation_ground_truth.csv"

    metrics = compute_val_metrics(
        model=model,
        composition_path=composition_path,
        ground_truth_path=ground_truth_path,
        data_dir=data_dir,
    )

    if metrics:
        mlflow.log_metrics(metrics)

        # gráfico de barras das métricas
        import seaborn as sns

        fig, ax = plt.subplots(figsize=(8, 5))
        metrics_df = pd.DataFrame(
            {
                "Metric": ["F1 (Samples)", "Precision (Samples)", "Recall (Samples)"],
                "Score": [
                    metrics["val_f1_samples"],
                    metrics["val_precision_samples"],
                    metrics["val_recall_samples"],
                ],
            }
        )
        sns.barplot(x="Metric", y="Score", data=metrics_df, ax=ax, palette="viridis")
        ax.set_ylim(0, 1.0)
        ax.set_title("Validation metrics — synthetic quadrats")
        for p in ax.patches:
            ax.annotate(
                f"{p.get_height():.3f}",
                (p.get_x() + p.get_width() / 2.0, p.get_height()),
                ha="center",
                va="bottom",
            )
        plot_path = "validation_metrics.png"
        fig.savefig(plot_path)
        mlflow.log_artifact(plot_path)
        plt.close(fig)

        print(
            f"Métricas → F1: {metrics['val_f1_samples']:.4f} | "
            f"Precision: {metrics['val_precision_samples']:.4f} | "
            f"Recall: {metrics['val_recall_samples']:.4f}"
        )

    print(f"\nRun concluído. MLflow UI: {os.environ['MLFLOW_TRACKING_URI']}")